In [1]:
import ssl
from urllib.request import Request, urlopen
from bs4 import BeautifulSoup
from openai import OpenAI

In [2]:
OLLAMA_BASE_URL = "http://localhost:11434/v1" 
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [3]:
def extract_website_text(url: str) -> str:
    
    ssl_context = ssl._create_unverified_context()
    headers = {"User-Agent": "Mozilla/5.0"}

    req = Request(url, headers=headers)
    html_data = urlopen(req, context=ssl_context).read().decode("utf-8")

    soup = BeautifulSoup(html_data, "html.parser")

    
    for element in soup(["script", "style", "nav", "footer", "header"]):
        element.decompose()

    
    paragraphs = [p.get_text(strip=True) for p in soup.find_all("p")]
    text = " ".join(paragraphs)

    return text

In [4]:

def summarize_webpage(url: str, model: str = "phi3") -> str:
    print(f"Fetching content from: {url}...")
    article_text = extract_website_text(url)

    
    truncated_text = article_text[:3000]
    system_prompt = (
        "You are a helpful assistant. Summarize the following website content clearly "
        "and concisely using key bullet points."
    )

    print(f"Generating summary using local model ({model})...")
    response = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": f"Please summarize this article:\n\n{truncated_text}",
            },
        ],
    )

    return response.choices[0].message.content

In [5]:
if __name__ == "__main__":
    target_url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
    summary = summarize_webpage(target_url, model="phi3")

    print("\n--- WEBPAGE SUMMARY ---")
    print(summary)

Fetching content from: https://en.wikipedia.org/wiki/Artificial_intelligence...
Generating summary using local model (phi3)...

--- WEBPAGE SUMMARY ---

- Artificial intelligence (AI) involves developing systems that can perform tasks usually associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making.
- High-profile AI applications include search engines, chatbots, virtual assistants, autonomous vehicles, and content generation, as well as strategy games like chess.
- Traditional AI research goals have been learning, reasoning, knowledge representation, planning, natural language processing, perception, and robotics.
- Researchers use techniques like state space search, mathematical optimization, formal logic, and artificial neural networks, along with statistics, operations research, and economics.
- AI also draws on psychology, linguistics, philosophy, neuroscience, and other fields.
- Companies like OpenAI, Google DeepMind, and 